## Predicting Dissolved Oxygen in Rivers

Data source: https://www.kaggle.com/datasets/vbmokin/dissolved-oxygen-prediction-in-river-water

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression, LinearRegression

In [2]:
data = pd.read_csv('archive/train.csv')
data

,Id,target,O2_1,O2_2,O2_3,O2_4,O2_5,O2_6,O2_7,NH4_1,...,NO3_5,NO3_6,NO3_7,BOD5_1,BOD5_2,BOD5_3,BOD5_4,BOD5_5,BOD5_6,BOD5_7
0,0,12.58,9.875,9.20,NaN,NaN,NaN,NaN,NaN,0.690,...,NaN,NaN,NaN,4.80,5.850,NaN,NaN,NaN,NaN,NaN
1,3,9.37,10.300,10.75,NaN,NaN,NaN,NaN,NaN,0.710,...,NaN,NaN,NaN,5.88,6.835,NaN,NaN,NaN,NaN,NaN
2,4,8.35,8.290,7.90,NaN,NaN,NaN,NaN,NaN,2.210,...,NaN,NaN,NaN,3.20,2.700,NaN,NaN,NaN,NaN,NaN
3,5,9.57,8.820,6.80,NaN,NaN,NaN,NaN,NaN,0.595,...,NaN,NaN,NaN,7.70,7.055,NaN,NaN,NaN,NaN,NaN
4,6,6.00,6.000,6.50,NaN,NaN,NaN,NaN,NaN,0.600,...,NaN,NaN,NaN,5.50,5.300,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,208,6.80,7.700,7.50,NaN,NaN,NaN,NaN,NaN,0.380,...,NaN,NaN,NaN,5.00,5.800,NaN,NaN,NaN,NaN,NaN
143,211,5.30,6.300,5.65,NaN,NaN,NaN,NaN,NaN,0.370,...,NaN,NaN,NaN,8.00,8.000,NaN,NaN,NaN,NaN,NaN
144,212,8.60,8.600,11.00,NaN,NaN,NaN,NaN,NaN,2.400,...,NaN,NaN,NaN,6.80,7.200,NaN,NaN,NaN,NaN,NaN
145,213,9.90,9.600,14.10,NaN,NaN,NaN,NaN,NaN,0.310,...,NaN,NaN,NaN,5.20,7.800,NaN,NaN,NaN,NaN,NaN


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 37 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Id      147 non-null    int64  
 1   target  147 non-null    float64
 2   O2_1    145 non-null    float64
 3   O2_2    145 non-null    float64
 4   O2_3    32 non-null     float64
 5   O2_4    31 non-null     float64
 6   O2_5    33 non-null     float64
 7   O2_6    37 non-null     float64
 8   O2_7    37 non-null     float64
 9   NH4_1   145 non-null    float64
 10  NH4_2   145 non-null    float64
 11  NH4_3   32 non-null     float64
 12  NH4_4   31 non-null     float64
 13  NH4_5   33 non-null     float64
 14  NH4_6   37 non-null     float64
 15  NH4_7   37 non-null     float64
 16  NO2_1   145 non-null    float64
 17  NO2_2   145 non-null    float64
 18  NO2_3   32 non-null     float64
 19  NO2_4   31 non-null     float64
 20  NO2_5   33 non-null     float64
 21  NO2_6   37 non-null     float64
 22  NO

### Preprocessing

#### Missing values

In [4]:
data.isna().sum()

Id          0
target      0
O2_1        2
O2_2        2
O2_3      115
O2_4      116
O2_5      114
O2_6      110
O2_7      110
NH4_1       2
NH4_2       2
NH4_3     115
NH4_4     116
NH4_5     114
NH4_6     110
NH4_7     110
NO2_1       2
NO2_2       2
NO2_3     115
NO2_4     116
NO2_5     114
NO2_6     110
NO2_7     110
NO3_1       2
NO3_2       2
NO3_3     115
NO3_4     116
NO3_5     114
NO3_6     110
NO3_7     110
BOD5_1      2
BOD5_2      2
BOD5_3    115
BOD5_4    116
BOD5_5    114
BOD5_6    110
BOD5_7    110
dtype: int64

In [5]:
null_columns = list(data.columns[data.isna().sum()>100])

data.drop(null_columns, axis=1, inplace=True)

In [6]:
data

,Id,target,O2_1,O2_2,NH4_1,NH4_2,NO2_1,NO2_2,NO3_1,NO3_2,BOD5_1,BOD5_2
0,0,12.58,9.875,9.20,0.690,1.040,0.0940,0.0990,1.58,1.825,4.80,5.850
1,3,9.37,10.300,10.75,0.710,0.725,0.0585,0.0515,1.21,0.905,5.88,6.835
2,4,8.35,8.290,7.90,2.210,2.210,0.1000,0.1100,1.34,1.250,3.20,2.700
3,5,9.57,8.820,6.80,0.595,0.675,0.0460,0.0535,0.59,0.790,7.70,7.055
4,6,6.00,6.000,6.50,0.600,0.900,0.1800,0.3400,1.36,1.820,5.50,5.300
...,...,...,...,...,...,...,...,...,...,...,...,...
142,208,6.80,7.700,7.50,0.380,1.900,0.6200,0.0640,2.80,3.330,5.00,5.800
143,211,5.30,6.300,5.65,0.370,0.500,0.6900,0.9500,4.37,3.160,8.00,8.000
144,212,8.60,8.600,11.00,2.400,3.600,0.1500,0.1400,0.53,3.000,6.80,7.200
145,213,9.90,9.600,14.10,0.310,0.500,0.2100,0.0800,3.10,3.500,5.20,7.800


In [7]:
data.isna().sum()

Id        0
target    0
O2_1      2
O2_2      2
NH4_1     2
NH4_2     2
NO2_1     2
NO2_2     2
NO3_1     2
NO3_2     2
BOD5_1    2
BOD5_2    2
dtype: int64

In [8]:
print("Columns with missing values:", (data.isna().sum(axis=0) != 0).sum())

Columns with missing values: 10


In [9]:
print("Rows with missing values:", (data.isna().sum(axis=1) != 0).sum())

Rows with missing values: 3


In [10]:
data.dropna(axis=0, inplace=True)

In [11]:
data.isna().sum().sum()

np.int64(0)

In [12]:
data.drop('Id', axis=1, inplace=True)

In [13]:
data

,target,O2_1,O2_2,NH4_1,NH4_2,NO2_1,NO2_2,NO3_1,NO3_2,BOD5_1,BOD5_2
0,12.58,9.875,9.20,0.690,1.040,0.0940,0.0990,1.58,1.825,4.80,5.850
1,9.37,10.300,10.75,0.710,0.725,0.0585,0.0515,1.21,0.905,5.88,6.835
2,8.35,8.290,7.90,2.210,2.210,0.1000,0.1100,1.34,1.250,3.20,2.700
3,9.57,8.820,6.80,0.595,0.675,0.0460,0.0535,0.59,0.790,7.70,7.055
4,6.00,6.000,6.50,0.600,0.900,0.1800,0.3400,1.36,1.820,5.50,5.300
...,...,...,...,...,...,...,...,...,...,...,...
142,6.80,7.700,7.50,0.380,1.900,0.6200,0.0640,2.80,3.330,5.00,5.800
143,5.30,6.300,5.65,0.370,0.500,0.6900,0.9500,4.37,3.160,8.00,8.000
144,8.60,8.600,11.00,2.400,3.600,0.1500,0.1400,0.53,3.000,6.80,7.200
145,9.90,9.600,14.10,0.310,0.500,0.2100,0.0800,3.10,3.500,5.20,7.800


#### Splitting and Scaling

In [14]:
y = data['target']
X = data.drop('target', axis=1)

In [15]:
scaler = StandardScaler()

X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [16]:
X

,O2_1,O2_2,NH4_1,NH4_2,NO2_1,NO2_2,NO3_1,NO3_2,BOD5_1,BOD5_2
0,0.179369,0.009505,0.219723,0.635946,-0.129370,-0.138744,-0.546590,-0.575052,-0.085090,0.464734
1,0.281515,0.423750,0.262989,0.101929,-0.433342,-0.344849,-0.672310,-0.973763,0.443285,0.960824
2,-0.201578,-0.337926,3.507976,2.619437,-0.077995,-0.091015,-0.628138,-0.824246,-0.867866,-1.121745
3,-0.074195,-0.631906,0.014207,0.017165,-0.540374,-0.336171,-0.882976,-1.023602,1.333693,1.071625
4,-0.751969,-0.712082,0.025023,0.398605,0.607012,0.906966,-0.621343,-0.577219,0.257375,0.187730
...,...,...,...,...,...,...,...,...,...,...
139,-0.343382,-0.444827,-0.450908,2.093896,4.374546,-0.290611,-0.132055,0.077186,0.012757,0.439552
140,-0.679865,-0.939249,-0.472541,-0.279511,4.973927,3.553785,0.401404,0.003511,1.480463,1.547569
141,-0.127071,0.490564,3.919008,4.975891,0.350134,0.039157,-0.903362,-0.065830,0.893381,1.144654
142,0.113274,1.319053,-0.602341,-0.279511,0.863889,-0.221186,-0.030120,0.150861,0.110605,1.446840


### Training (Regression)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=42)

In [18]:
model = LinearRegression()
model.fit(X_train, y_train)

model_R2 = model.score(X_test, y_test)

print("Linear Regression R2 Score: {:.4f}".format(model_R2))

Linear Regression R2 Score: 0.3177


### New Problem: Predicting High or Low Dissolved Oxygen

In [19]:
y

0      12.58
1       9.37
2       8.35
3       9.57
4       6.00
       ...  
142     6.80
143     5.30
144     8.60
145     9.90
146     6.50
Name: target, Length: 144, dtype: float64

In [20]:
y_new = pd.qcut(y, q=2, labels=[0,1])
y_new

0      1
1      1
2      0
3      1
4      0
      ..
142    0
143    0
144    0
145    1
146    0
Name: target, Length: 144, dtype: category
Categories (2, int64): [0 < 1]

### Training (Classification)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y_new, train_size=0.7, random_state=42)

In [22]:
model = LogisticRegression()
model.fit(X_train, y_train)

model_acc = model.score(X_test, y_test)

print("Model Accuracy: {:.3f}".format(model_acc))

Model Accuracy: 0.818
